In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw_data"

sales = pd.read_csv(
    DATA_DIR / "sales_info" / "서울시(추정매출-행정동).csv",
    encoding="cp949"
)

store = pd.read_csv(
    DATA_DIR / "store_info" / "상가(상권)정보_서울.csv",
    encoding="utf-8"
)

population = pd.read_csv(
    DATA_DIR / "population_info" / "서울시(유동인구-행정동).csv",
    encoding="cp949"
)

print("파일 불러오기 완료")

In [2]:
#행정동코드로 컬럼명 통일
sales = sales.rename(columns={"행정동_코드": "행정동코드"})
population = population.rename(columns={"행정동_코드": "행정동코드"})

In [3]:
#컬럼 명 앞뒤 공백 제거
sales.columns = sales.columns.str.strip()
store.columns = store.columns.str.strip()
population.columns = population.columns.str.strip()

In [4]:
#타입 통일화
sales["행정동코드"] = sales["행정동코드"].astype(str).str.strip()
population["행정동코드"] = population["행정동코드"].astype(str).str.strip()
store["행정동코드"] = store["행정동코드"].astype(str).str.strip()

In [5]:
print("sales keys:", "기준_년분기_코드" in sales.columns, "행정동코드" in sales.columns, "서비스_업종_코드_명" in sales.columns)
print("store keys:", "행정동코드" in store.columns, "상권업종중분류명" in store.columns)
print("pop keys:", "기준_년분기_코드" in population.columns, "행정동코드" in population.columns, "행정동_코드_명" in population.columns)

sales keys: True True True
store keys: True True
pop keys: True True True


In [6]:
# 통합 카테고리 맵핑
SALES_CATEGORY_MAP = {
    # 음식/식음료
    "한식음식점": "한식",
    "분식전문점": "분식/간식",
    "제과점": "분식/간식",
    "중식음식점": "중식",
    "일식음식점": "일식",
    "양식음식점": "양식/기타외식",
    "커피-음료": "카페",
    "치킨전문점": "패스트푸드/치킨",
    "패스트푸드점": "패스트푸드/치킨",
    "호프-간이주점": "주점",
    # 소매
    "편의점": "편의점",
    "슈퍼마켓": "식품 소매",
    "미곡판매": "식품 소매",
    "수산물판매": "식품 소매",
    "육류판매": "식품 소매",
    "청과상": "식품 소매",
    "반찬가게": "식품 소매",
    "일반의류": "의류/패션",
    "신발": "의류/패션",
    "가방": "의류/패션",
    "섬유제품": "의류/패션",
    "가전제품": "전자/통신",
    "핸드폰": "전자/통신",
    "컴퓨터및주변장치판매": "전자/통신",
    "전자상거래업": "전자/통신",
    "가구": "생활용품 소매",
    "조명용품": "생활용품 소매",
    "철물점": "생활용품 소매",
    "문구": "생활용품 소매",
    "완구": "생활용품 소매",
    "화초": "생활용품 소매",
    "운동/경기용품": "생활용품 소매",
    "서적": "생활용품 소매",
    "시계및귀금속": "생활용품 소매",
    "화장품": "뷰티/화장품",
    "네일숍": "뷰티/화장품",
    "피부관리실": "뷰티/화장품",
    "애완동물": "애완동물",
    # 생활 서비스
    "미용실": "미용실",
    "일반의원": "의료/약국",
    "치과의원": "의료/약국",
    "한의원": "의료/약국",
    "의료기기": "의료/약국",
    "의약품": "의료/약국",
    "안경": "의료/약국",
    "외국어학원": "일반학원",
    "일반교습학원": "일반학원",
    "예술학원": "예술학원",
    "스포츠 강습": "스포츠 강습",
    "골프연습장": "스포츠/레저",
    "스포츠클럽": "스포츠/레저",
    "PC방": "오락/유흥",
    "노래방": "오락/유흥",
    "당구장": "오락/유흥",
    "여관": "숙박",
    "고시원": "숙박",
    "가전제품수리": "수리/세탁",
    "세탁소": "수리/세탁",
    # B2B
    "자동차수리": "B2B 서비스",
    "자동차미용": "B2B 서비스",
    "자전거 및 기타운송장비": "B2B 서비스",
    "부동산중개업": "B2B 서비스",
    "인테리어": "B2B 서비스",
}

STORE_CATEGORY_MAP = {
    # 음식/식음료
    "한식": "한식",
    "구내식당·뷔페": "한식",
    "기타 간이": "분식/간식",
    "중식": "중식",
    "동남아시아": "중식",
    "일식": "일식",
    "서양식": "양식/기타외식",
    "기타 외국": "양식/기타외식",
    "비알코올": "카페",
    "주점": "주점",
    # 소매
    "종합 소매": "편의점",
    "식료품 소매": "식품 소매",
    "음료 소매": "식품 소매",
    "담배 소매": "식품 소매",
    "섬유·의복·신발 소매": "의류/패션",
    "시계·귀금속 소매": "의류/패션",
    "중고 상품 소매": "의류/패션",
    "가전·통신 소매": "전자/통신",
    "가구 소매": "생활용품 소매",
    "기타 생활용품 소매": "생활용품 소매",
    "오락용품 소매": "생활용품 소매",
    "식물 소매": "생활용품 소매",
    "장식품 소매": "생활용품 소매",
    "안경·정밀기기 소매": "생활용품 소매",
    "기타 상품 소매": "생활용품 소매",
    "의약·화장품 소매": "뷰티/화장품",
    "욕탕·신체관리": "뷰티/화장품",
    "애완동물·용품 소매": "애완동물",
    # 생활 서비스
    "이용·미용": "미용실",
    "의원": "의료/약국",
    "병원": "의료/약국",
    "기타 보건": "의료/약국",
    "수의": "의료/약국",
    "일반 교육": "일반학원",
    "교육 지원": "일반학원",
    "기타 교육": "예술학원",
    "스포츠 서비스": "스포츠/레저",
    "유원지·오락": "오락/유흥",
    "일반 숙박": "숙박",
    "기타 숙박": "숙박",
    "가전제품 수리": "수리/세탁",
    "세탁": "수리/세탁",
    "기타 가정용품 수리": "수리/세탁",
    "컴퓨터 수리": "수리/세탁",
    "통신장비 수리": "수리/세탁",
    # B2B
    "자동차 수리·세차": "B2B 서비스",
    "자동차 부품 소매": "B2B 서비스",
    "모터사이클 소매": "B2B 서비스",
    "모터사이클 수리": "B2B 서비스",
    "부동산 서비스": "B2B 서비스",
    "철물·건설자재 소매": "B2B 서비스",
    "광고": "B2B 서비스",
    "기술 서비스": "B2B 서비스",
    "법무관련": "B2B 서비스",
    "회계·세무": "B2B 서비스",
    "사무 지원": "B2B 서비스",
    "전문 디자인": "B2B 서비스",
    "본사·경영 컨설팅": "B2B 서비스",
    "인쇄·제품제작": "B2B 서비스",
    "시장 조사": "B2B 서비스",
    "조경·유지": "B2B 서비스",
    "청소·방제": "B2B 서비스",
    "사진 촬영": "B2B 서비스",
    "고용 알선": "B2B 서비스",
    "여행사·보조": "B2B 서비스",
    "기타 사업 서비스": "B2B 서비스",
    "기타 전문 과학": "B2B 서비스",
    "시설관리": "B2B 서비스",
    "기타 개인": "B2B 서비스",
    "산업용품 대여": "B2B 서비스",
    "운송장비 대여": "B2B 서비스",
    "가정용품 대여": "B2B 서비스",
    "연료 소매": "B2B 서비스",
    "장례식장": "B2B 서비스",
    "도서관·사적지": "B2B 서비스",
}

# 공백 정리 후 매핑
sales["서비스_업종_코드_명"] = sales["서비스_업종_코드_명"].astype(str).str.strip()
store["상권업종중분류명"] = store["상권업종중분류명"].astype(str).str.strip()
store["상권업종소분류명"] = store["상권업종소분류명"].astype(str).str.strip()

sales["통합카테고리"] = sales["서비스_업종_코드_명"].map(SALES_CATEGORY_MAP)
store["통합카테고리"] = store["상권업종중분류명"].map(STORE_CATEGORY_MAP)

# 소분류 기반 재매핑: "기타 간이" 중 치킨/피자 → "패스트푸드/치킨"
fastfood_mask = (
    (store["상권업종중분류명"] == "기타 간이") &
    (store["상권업종소분류명"].isin(["치킨", "피자"]))
)
store.loc[fastfood_mask, "통합카테고리"] = "패스트푸드/치킨"

print("성공")
print(f"패스트푸드/치킨으로 재매핑된 점포 수: {fastfood_mask.sum()}")

성공
패스트푸드/치킨으로 재매핑된 점포 수: 8445


In [7]:
#매핑 안된 컬럼 확인
print("sales 매핑 안된 개수:", sales["통합카테고리"].isna().sum())
print("store 매핑 안된 개수:", store["통합카테고리"].isna().sum())

sales 매핑 안된 개수: 0
store 매핑 안된 개수: 0


sales 필요한 컬럼 뽑기

In [8]:
# 필요한 컬럼만 선택
sales_needed_cols = [
    "기준_년분기_코드",
    "행정동코드",
    "통합카테고리",
    "당월_매출_금액",
    "연령대_20_매출_금액",
]

sales_small = sales[sales_needed_cols].copy()

# 숫자 컬럼이 문자열로 들어왔을 때 대비 (에러 없이 숫자로 변환)
num_cols = ["당월_매출_금액", "연령대_20_매출_금액"]
for c in num_cols:
    sales_small[c] = pd.to_numeric(sales_small[c], errors="coerce")

# 결측은 0으로 (집계 안정화)
sales_small[num_cols] = sales_small[num_cols].fillna(0)

print("sales_small shape:", sales_small.shape)
sales_small.head()

sales_small shape: (50395, 5)


,기준_년분기_코드,행정동코드,통합카테고리,당월_매출_금액,연령대_20_매출_금액
0,20253,11740700,전자/통신,10751618,0
1,20253,11740700,생활용품 소매,8249940,76708
2,20253,11740700,B2B 서비스,661900993,5562102
3,20253,11740700,생활용품 소매,115789484,227558
4,20253,11740700,생활용품 소매,13984669,0


In [9]:
sales_grouped = (
    sales_small
    .groupby(["기준_년분기_코드", "행정동코드", "통합카테고리"], as_index=False)
    .agg(
        당월매출합=("당월_매출_금액", "sum"),
        매출_20대합=("연령대_20_매출_금액", "sum")
    )
)

print("sales_grouped shape:", sales_grouped.shape)
sales_grouped.head()

sales_grouped shape: (27163, 5)


,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합
0,20251,11110515,B2B 서비스,268180168,848886
1,20251,11110515,미용실,261655455,18201326
2,20251,11110515,분식/간식,876235565,130438123
3,20251,11110515,뷰티/화장품,18740937,0
4,20251,11110515,생활용품 소매,4485007089,2128861032


population 필요한 컬럼 뽑기

In [10]:

# 필요한 컬럼만 선택
pop_needed_cols = [
    "기준_년분기_코드",
    "행정동코드",
    "행정동_코드_명",     
    "총_유동인구_수",
    "연령대_20_유동인구_수",
]

population_small = population[pop_needed_cols].copy()

# 숫자 컬럼 안전 변환 + 결측 0
pop_num_cols = ["총_유동인구_수", "연령대_20_유동인구_수"]
for c in pop_num_cols:
    population_small[c] = pd.to_numeric(population_small[c], errors="coerce")
population_small[pop_num_cols] = population_small[pop_num_cols].fillna(0)

population_small.head()

,기준_년분기_코드,행정동코드,행정동_코드_명,총_유동인구_수,연령대_20_유동인구_수
0,20253,11740700,둔촌2동,6677641,759362
1,20253,11740690,둔촌1동,30002,2516
2,20253,11740685,길동,18306303,2213064
3,20253,11740660,성내3동,6680704,886276
4,20253,11740650,성내2동,8182706,1132456


In [11]:
population_grouped = (
    population_small
    .groupby(["기준_년분기_코드", "행정동코드"], as_index=False)
    .agg(
        행정동명=("행정동_코드_명", "first"),
        총유동인구=("총_유동인구_수", "sum"),
        유동_20대=("연령대_20_유동인구_수", "sum")
    )
)

population_grouped.head()

,기준_년분기_코드,행정동코드,행정동명,총유동인구,유동_20대
0,20191,11110515,청운효자동,3668225,542715
1,20191,11110530,사직동,4628956,752637
2,20191,11110540,삼청동,1010831,152090
3,20191,11110550,부암동,1053668,143980
4,20191,11110560,평창동,1139144,119637


sales+population 합치기

In [12]:
sales_pop = sales_grouped.merge(
    population_grouped,
    on=["기준_년분기_코드", "행정동코드"],
    how="left"
)

print("sales_pop shape:", sales_pop.shape)
sales_pop.head()

sales_pop shape: (27163, 8)


,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합,행정동명,총유동인구,유동_20대
0,20251,11110515,B2B 서비스,268180168,848886,청운효자동,3322244,442023
1,20251,11110515,미용실,261655455,18201326,청운효자동,3322244,442023
2,20251,11110515,분식/간식,876235565,130438123,청운효자동,3322244,442023
3,20251,11110515,뷰티/화장품,18740937,0,청운효자동,3322244,442023
4,20251,11110515,생활용품 소매,4485007089,2128861032,청운효자동,3322244,442023


store grouped 시키기

In [13]:
store_grouped = (
    store
    .groupby(["행정동코드", "통합카테고리"], as_index=False)
    .size()
    .rename(columns={"size": "점포수"})
)

print("store_grouped shape:", store_grouped.shape)
store_grouped.head()

store_grouped shape: (10085, 3)


,행정동코드,통합카테고리,점포수
0,11110515,B2B 서비스,256
1,11110515,미용실,24
2,11110515,분식/간식,40
3,11110515,뷰티/화장품,12
4,11110515,생활용품 소매,69


salse_pop+store 합치기

In [14]:
final = sales_pop.merge(
    store_grouped,
    on=["행정동코드", "통합카테고리"],
    how="left"
)

print("final shape:", final.shape)
final.head()

final shape: (27163, 9)


,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합,행정동명,총유동인구,유동_20대,점포수
0,20251,11110515,B2B 서비스,268180168,848886,청운효자동,3322244,442023,256.0
1,20251,11110515,미용실,261655455,18201326,청운효자동,3322244,442023,24.0
2,20251,11110515,분식/간식,876235565,130438123,청운효자동,3322244,442023,40.0
3,20251,11110515,뷰티/화장품,18740937,0,청운효자동,3322244,442023,12.0
4,20251,11110515,생활용품 소매,4485007089,2128861032,청운효자동,3322244,442023,69.0


점포수 누락 확인 및 0으로 처리

In [15]:
print("점포수 NaN 개수:", final["점포수"].isna().sum())

final["점포수"] = final["점포수"].fillna(0).astype(int)

print("점포수 NaN(채운 후):", final["점포수"].isna().sum())
final.head()

점포수 NaN 개수: 1298
점포수 NaN(채운 후): 0


,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합,행정동명,총유동인구,유동_20대,점포수
0,20251,11110515,B2B 서비스,268180168,848886,청운효자동,3322244,442023,256
1,20251,11110515,미용실,261655455,18201326,청운효자동,3322244,442023,24
2,20251,11110515,분식/간식,876235565,130438123,청운효자동,3322244,442023,40
3,20251,11110515,뷰티/화장품,18740937,0,청운효자동,3322244,442023,12
4,20251,11110515,생활용품 소매,4485007089,2128861032,청운효자동,3322244,442023,69


=====파생 변수 생성=====

In [16]:
# 행정동 전체 매출
dong_total_sales = final.groupby(
    ["기준_년분기_코드", "행정동코드"]
)["당월매출합"].transform("sum")

final["행정동_전체매출"] = dong_total_sales

final.head()

,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합,행정동명,총유동인구,유동_20대,점포수,행정동_전체매출
0,20251,11110515,B2B 서비스,268180168,848886,청운효자동,3322244,442023,256,23027279079
1,20251,11110515,미용실,261655455,18201326,청운효자동,3322244,442023,24,23027279079
2,20251,11110515,분식/간식,876235565,130438123,청운효자동,3322244,442023,40,23027279079
3,20251,11110515,뷰티/화장품,18740937,0,청운효자동,3322244,442023,12,23027279079
4,20251,11110515,생활용품 소매,4485007089,2128861032,청운효자동,3322244,442023,69,23027279079


In [17]:
# 행정동 전체 점포수
dong_total_store = final.groupby(
    ["기준_년분기_코드", "행정동코드"]
)["점포수"].transform("sum")

final["행정동_전체점포수"] = dong_total_store

final.head()

,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합,행정동명,총유동인구,유동_20대,점포수,행정동_전체매출,행정동_전체점포수
0,20251,11110515,B2B 서비스,268180168,848886,청운효자동,3322244,442023,256,23027279079,828
1,20251,11110515,미용실,261655455,18201326,청운효자동,3322244,442023,24,23027279079,828
2,20251,11110515,분식/간식,876235565,130438123,청운효자동,3322244,442023,40,23027279079,828
3,20251,11110515,뷰티/화장품,18740937,0,청운효자동,3322244,442023,12,23027279079,828
4,20251,11110515,생활용품 소매,4485007089,2128861032,청운효자동,3322244,442023,69,23027279079,828


In [18]:
# 1️⃣ 업종 점포당 매출
final["업종_점포당매출"] = final["당월매출합"] / final["점포수"].replace(0, 1)


# 2️⃣ 업종 매출 점유율
final["업종_매출점유율"] = final["당월매출합"] / final["행정동_전체매출"]


# 3️⃣ 업종 포화도
final["업종_포화도"] = final["점포수"] / final["행정동_전체점포수"]


# 4️⃣ 경쟁 강도 (점포 밀도 개념)
final["경쟁강도"] = final["점포수"] / final["총유동인구"]


# 5️⃣ MZ 비율
final["매출_20대비율"] = final["매출_20대합"] / final["당월매출합"].replace(0, 1)
final["유동_20대비율"] = final["유동_20대"] / final["총유동인구"]

final["MZ_차이"] = final["매출_20대비율"] - final["유동_20대비율"]


final.head()

,기준_년분기_코드,행정동코드,통합카테고리,당월매출합,매출_20대합,행정동명,총유동인구,유동_20대,점포수,행정동_전체매출,행정동_전체점포수,업종_점포당매출,업종_매출점유율,업종_포화도,경쟁강도,매출_20대비율,유동_20대비율,MZ_차이
0,20251,11110515,B2B 서비스,268180168,848886,청운효자동,3322244,442023,256,23027279079,828,1.047579e+06,0.011646,0.309179,0.000077,0.003165,0.13305,-0.129884
1,20251,11110515,미용실,261655455,18201326,청운효자동,3322244,442023,24,23027279079,828,1.090231e+07,0.011363,0.028986,0.000007,0.069562,0.13305,-0.063487
2,20251,11110515,분식/간식,876235565,130438123,청운효자동,3322244,442023,40,23027279079,828,2.190589e+07,0.038052,0.048309,0.000012,0.148862,0.13305,0.015812
3,20251,11110515,뷰티/화장품,18740937,0,청운효자동,3322244,442023,12,23027279079,828,1.561745e+06,0.000814,0.014493,0.000004,0.000000,0.13305,-0.133050
4,20251,11110515,생활용품 소매,4485007089,2128861032,청운효자동,3322244,442023,69,23027279079,828,6.500010e+07,0.194769,0.083333,0.000021,0.474662,0.13305,0.341612


======추가 파생 변수=====

In [19]:
# 유동대비매출(상권 구매력)
final["유동대비매출"] = final["당월매출합"] / final["총유동인구"].replace(0, 1)

# 점포대비유동(점포 경쟁력)
final["점포대비유동"] = final["총유동인구"] / final["점포수"].replace(0, 1)

final[["당월매출합","총유동인구","점포수","유동대비매출","점포대비유동"]].head()

,당월매출합,총유동인구,점포수,유동대비매출,점포대비유동
0,268180168,3322244,256,80.722598,12977.515625
1,261655455,3322244,24,78.758651,138426.833333
2,876235565,3322244,40,263.748107,83056.100000
3,18740937,3322244,12,5.641048,276853.666667
4,4485007089,3322244,69,1349.993284,48148.463768


결측치 확인 및 해결

In [20]:
final.isna().sum()

기준_년분기_코드      0
행정동코드          0
통합카테고리         0
당월매출합          0
매출_20대합        0
행정동명           0
총유동인구          0
유동_20대         0
점포수            0
행정동_전체매출       0
행정동_전체점포수      0
업종_점포당매출       0
업종_매출점유율       0
업종_포화도       112
경쟁강도           0
매출_20대비율       0
유동_20대비율       0
MZ_차이          0
유동대비매출         0
점포대비유동         0
dtype: int64

In [21]:
final.loc[final["업종_포화도"].isna(), ["기준_년분기_코드","행정동코드","통합카테고리","점포수","행정동_전체점포수"]].head(10)

,기준_년분기_코드,행정동코드,통합카테고리,점포수,행정동_전체점포수
8063,20251,11680740,미용실,0,0
8064,20251,11680740,분식/간식,0,0
8065,20251,11680740,뷰티/화장품,0,0
8066,20251,11680740,생활용품 소매,0,0
8067,20251,11680740,수리/세탁,0,0
8068,20251,11680740,스포츠 강습,0,0
8069,20251,11680740,식품 소매,0,0
8070,20251,11680740,예술학원,0,0
8071,20251,11680740,의료/약국,0,0
8072,20251,11680740,일반학원,0,0


In [22]:
(final["행정동_전체점포수"] == 0).sum()

np.int64(112)

In [23]:
final["업종_포화도"] = final["점포수"] / final["행정동_전체점포수"].replace(0, 1)
print("업종_포화도 NaN:", final["업종_포화도"].isna().sum())

업종_포화도 NaN: 0


In [24]:
(final["업종_포화도"] == 0).sum()

np.int64(1298)

In [25]:
import numpy as np

np.isinf(final.select_dtypes(include=[np.number])).sum()

기준_년분기_코드    0
당월매출합        0
매출_20대합      0
총유동인구        0
유동_20대       0
점포수          0
행정동_전체매출     0
행정동_전체점포수    0
업종_점포당매출     0
업종_매출점유율     0
업종_포화도       0
경쟁강도         0
매출_20대비율     0
유동_20대비율     0
MZ_차이        0
유동대비매출       0
점포대비유동       0
dtype: int64

In [26]:
final.describe()

,기준_년분기_코드,당월매출합,매출_20대합,총유동인구,유동_20대,점포수,행정동_전체매출,행정동_전체점포수,업종_점포당매출,업종_매출점유율,업종_포화도,경쟁강도,매출_20대비율,유동_20대비율,MZ_차이,유동대비매출,점포대비유동
count,27163.000000,2.716300e+04,2.716300e+04,2.716300e+04,2.716300e+04,27163.000000,2.716300e+04,27163.000000,2.716300e+04,2.716300e+04,27163.000000,27163.000000,27163.000000,27163.000000,27163.000000,27163.000000,2.716300e+04
mean,20252.001730,2.819121e+09,2.335352e+08,5.658023e+06,9.157741e+05,56.661819,6.420573e+10,1302.385414,7.099229e+07,4.693885e-02,0.046718,0.000011,0.090944,0.151717,-0.060773,651.233459,5.288101e+05
std,0.816607,1.610883e+10,1.036471e+09,3.015776e+06,7.405625e+05,163.416183,9.679727e+10,1371.594517,2.782938e+08,8.755371e-02,0.058530,0.000033,0.095211,0.060983,0.088421,7428.110585,1.458072e+06
min,20251.000000,3.031600e+04,0.000000e+00,2.323900e+04,1.974000e+03,0.000000,7.489562e+07,0.000000,4.976220e+02,4.214027e-07,0.000000,0.000000,0.000000,0.068733,-0.459188,0.006049,7.775761e+02
25%,20251.000000,1.473399e+08,4.728836e+06,3.604843e+06,4.450620e+05,14.000000,2.040871e+10,604.000000,6.906410e+06,4.655095e-03,0.016295,0.000003,0.022836,0.113758,-0.109928,32.115171,9.656223e+04
50%,20252.000000,5.148534e+08,3.100931e+07,5.238510e+06,7.293240e+05,28.000000,3.666926e+10,940.000000,2.100000e+07,1.407880e-02,0.031182,0.000006,0.065124,0.134676,-0.070806,102.164357,1.762659e+05
75%,20253.000000,1.748121e+09,1.437979e+08,6.921656e+06,1.165388e+06,54.000000,6.548084e+10,1445.000000,5.567930e+07,4.049087e-02,0.051975,0.000010,0.127260,0.166937,-0.020134,349.923998,3.282095e+05
max,20253.000000,9.294219e+11,4.489299e+10,2.232899e+07,5.838683e+06,6665.000000,9.844805e+11,13380.000000,1.777370e+10,9.636436e-01,0.679245,0.001286,1.000000,0.476355,0.904604,577126.884083,2.232899e+07


In [27]:
final.describe().T

,count,mean,std,min,25%,50%,75%,max
기준_년분기_코드,27163.0,2.025200e+04,8.166075e-01,2.025100e+04,2.025100e+04,2.025200e+04,2.025300e+04,2.025300e+04
당월매출합,27163.0,2.819121e+09,1.610883e+10,3.031600e+04,1.473399e+08,5.148534e+08,1.748121e+09,9.294219e+11
매출_20대합,27163.0,2.335352e+08,1.036471e+09,0.000000e+00,4.728836e+06,3.100931e+07,1.437979e+08,4.489299e+10
총유동인구,27163.0,5.658023e+06,3.015776e+06,2.323900e+04,3.604843e+06,5.238510e+06,6.921656e+06,2.232899e+07
유동_20대,27163.0,9.157741e+05,7.405625e+05,1.974000e+03,4.450620e+05,7.293240e+05,1.165388e+06,5.838683e+06
점포수,27163.0,5.666182e+01,1.634162e+02,0.000000e+00,1.400000e+01,2.800000e+01,5.400000e+01,6.665000e+03
행정동_전체매출,27163.0,6.420573e+10,9.679727e+10,7.489562e+07,2.040871e+10,3.666926e+10,6.548084e+10,9.844805e+11
행정동_전체점포수,27163.0,1.302385e+03,1.371595e+03,0.000000e+00,6.040000e+02,9.400000e+02,1.445000e+03,1.338000e+04
업종_점포당매출,27163.0,7.099229e+07,2.782938e+08,4.976220e+02,6.906410e+06,2.100000e+07,5.567930e+07,1.777370e+10
업종_매출점유율,27163.0,4.693885e-02,8.755371e-02,4.214027e-07,4.655095e-03,1.407880e-02,4.049087e-02,9.636436e-01


In [28]:
len(final)

27163

In [29]:
final.columns

Index(['기준_년분기_코드', '행정동코드', '통합카테고리', '당월매출합', '매출_20대합', '행정동명', '총유동인구',
       '유동_20대', '점포수', '행정동_전체매출', '행정동_전체점포수', '업종_점포당매출', '업종_매출점유율',
       '업종_포화도', '경쟁강도', '매출_20대비율', '유동_20대비율', 'MZ_차이', '유동대비매출', '점포대비유동'],
      dtype='object')

In [ ]:
final.to_csv("../outputs/final_dataset.csv", index=False, encoding="utf-8-sig")